# Unsloth Qwen3-1.7B 简洁实验 Notebook

这个 notebook 只保留最常用、最清晰的流程：

1. **指定 GPU**（必须在 import torch / unsloth 之前）。
2. 从本地 `model/Qwen3-1.7B` 加载模型。
3. 注入 LoRA / QLoRA adapter。
4. SFT 训练并保存 adapter。
5. 单条推理与批量推理。
6. 可选：GRPO/RL。

> 如果你只做 SFT + batch infer，运行到第 8 节即可；GRPO 是可选部分。

## 0. 指定 GPU（最重要）

Unsloth / PyTorch 会把 `CUDA_VISIBLE_DEVICES` 看到的第一个 GPU 当作 `cuda:0`。

推荐方式是用仓库启动脚本指定物理 GPU：

```bash
python scripts/launch_notebook.py --gpu_id 1
```

这会在启动 notebook 前设置 `CUDA_VISIBLE_DEVICES=1`，进入 notebook 后物理 1 号卡会显示为 `cuda:0`。

如果已经打开 notebook，也可以在下面 cell 里设置；但必须保证这是 kernel 启动后第一个执行的 Python cell，且之前没有 import 过 `torch` / `unsloth`。如果你已经运行过后面的 cell，请 **Restart Kernel** 后再改这里。

In [ ]:
import os

# 如果你不是用 scripts/launch_notebook.py --gpu_id 启动，才需要改这里。
# 物理 GPU 号例如 "0"、"1"、"2"；进程内会重新编号为 cuda:0。
SELECTED_GPU = "0"
os.environ["CUDA_VISIBLE_DEVICES"] = SELECTED_GPU
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# 减少显存碎片导致的 OOM；必须在 import torch 前设置。
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

print("CUDA_VISIBLE_DEVICES =", os.environ["CUDA_VISIBLE_DEVICES"])

## 1. 实验配置

你已经下载好的模型默认放在仓库根目录：`model/Qwen3-1.7B`。

In [ ]:
from pathlib import Path
import sys

# 自动识别仓库根目录：既支持从 repo root 打开，也支持从 notebooks/ 打开。
REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / "src").exists() and (REPO_ROOT.parent / "src").exists():
    REPO_ROOT = REPO_ROOT.parent.resolve()
sys.path.insert(0, str(REPO_ROOT / "src"))

# 本地模型与数据
MODEL_NAME_OR_PATH = str(REPO_ROOT / "model" / "Qwen3-1.7B")
TRAIN_FILE = REPO_ROOT / "data" / "toy_sft.jsonl"
PROMPT_FILE = REPO_ROOT / "data" / "prompts.json"

# 输出目录
SFT_OUTPUT_DIR = REPO_ROOT / "outputs" / "qwen3_1p7b_unsloth_lora"
BATCH_OUTPUT_FILE = REPO_ROOT / "outputs" / "qwen3_1p7b_batch_outputs.json"
GRPO_OUTPUT_DIR = REPO_ROOT / "outputs" / "qwen3_1p7b_unsloth_grpo"

# 数据字段
PROMPT_FIELD = "prompt"
RESPONSE_FIELD = "groundtruth"
MESSAGES_FIELD = "messages"

# 序列长度与量化
MAX_SEQ_LENGTH = 512      # 3090 上先用 512 跑通；显存充足再调到 1024/2048
LOAD_IN_4BIT = True     # True=QLoRA；False=普通 LoRA
DTYPE = None            # None 让 Unsloth 自动选择；也可设 torch.float16 / torch.bfloat16

# LoRA 超参（Qwen 常用 target modules）
LORA_R = 16
LORA_ALPHA = 16
LORA_DROPOUT = 0.0
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]

# SFT 超参
RESPONSE_ONLY_LOSS = True   # True=只训练 assistant answer；False=prompt+answer 都算 loss
NUM_TRAIN_EPOCHS = 1
LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4

# 推理参数
MAX_NEW_TOKENS = 64
TEMPERATURE = 0.7
TOP_P = 0.9
INFER_BATCH_SIZE = 1
NUM_REPEATS = 1
MIN_FREE_GPU_MEMORY_GB = 6.0  # 如果可见 GPU 空闲显存低于此值，提前报错而不是等 OOM

print("Repo root:", REPO_ROOT)
print("Model path:", MODEL_NAME_OR_PATH)
print("Model exists:", Path(MODEL_NAME_OR_PATH).exists())

## 2. 导入依赖并检查 CUDA

In [ ]:
import json
from typing import Any

import torch
from unsloth import FastLanguageModel, is_bfloat16_supported

from llm_lab.data import _apply_chat_template, load_sft_dataset, load_grpo_dataset
from llm_lab.model_utils import ensure_pad_token, print_cuda_info, require_min_cuda_memory
from llm_lab.train_utils import ResponseOnlyDataCollator, build_sft_trainer, get_training_args

print("PyTorch:", torch.__version__)
print("PyTorch CUDA build:", torch.version.cuda)
print_cuda_info()
require_min_cuda_memory(MIN_FREE_GPU_MEMORY_GB, context="notebook Unsloth run")

if not torch.cuda.is_available():
    raise RuntimeError("CUDA 不可用。请检查驱动、PyTorch CUDA wheel、CUDA_VISIBLE_DEVICES。")
if not Path(MODEL_NAME_OR_PATH).exists():
    raise FileNotFoundError(f"未找到模型目录：{MODEL_NAME_OR_PATH}")
print("Current CUDA device:", torch.cuda.current_device(), torch.cuda.get_device_name(0))

### 如果这里提示显存不足怎么办？

如果报错类似 `GPU 0 only has 0.30 GiB free`，说明 notebook 当前可见的 GPU 已经被占用。请不要继续调 batch size，先处理 GPU：

1. 在终端运行 `nvidia-smi` 看哪张物理卡空闲。
2. 重启 kernel，并在第 0 个 cell 把 `SELECTED_GPU` 改成空闲卡号；或用 `CUDA_VISIBLE_DEVICES=空闲卡号 jupyter notebook ...` 启动。
3. 确认第 2 节打印的 free memory 足够后再加载模型。

当前 notebook 已把 `MAX_SEQ_LENGTH=512`、`INFER_BATCH_SIZE=1` 作为保守默认值，方便先跑通。

## 3. 加载模型并注入 LoRA

这部分基本等同于 Unsloth 官方文档里的核心代码。

In [ ]:
import gc

gc.collect()
torch.cuda.empty_cache()

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME_OR_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype = DTYPE,
    load_in_4bit = LOAD_IN_4BIT,
)

ensure_pad_token(tokenizer)
tokenizer.padding_side = "left"

model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_R,
    target_modules = TARGET_MODULES,
    lora_alpha = LORA_ALPHA,
    lora_dropout = LORA_DROPOUT,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
    use_rslora = False,
    loftq_config = None,
)

## 4. 加载 SFT 数据

支持三种格式：

```json
{"prompt": "...", "groundtruth": "..."}
{"messages": [{"role": "user", "content": "..."}, {"role": "assistant", "content": "..."}]}
{"text": "已经渲染好的训练文本"}
```

In [ ]:
train_dataset = load_sft_dataset(
    TRAIN_FILE,
    tokenizer,
    prompt_field = PROMPT_FIELD,
    response_field = RESPONSE_FIELD,
    response_only_loss = RESPONSE_ONLY_LOSS,
)

print(train_dataset)
print("Columns:", train_dataset.column_names)
print("First row keys:", train_dataset[0].keys())

## 5. SFT 训练

- `RESPONSE_ONLY_LOSS=True`：使用 `Trainer + ResponseOnlyDataCollator`，只对 assistant answer 计算 loss。
- `RESPONSE_ONLY_LOSS=False`：使用 TRL `SFTTrainer`，对完整文本计算 loss。

In [ ]:
from transformers import Trainer

fp16 = not is_bfloat16_supported()
bf16 = is_bfloat16_supported()

training_args = get_training_args(
    output_dir = str(SFT_OUTPUT_DIR),
    max_length = MAX_SEQ_LENGTH,
    num_train_epochs = NUM_TRAIN_EPOCHS,
    learning_rate = LEARNING_RATE,
    per_device_train_batch_size = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    fp16 = fp16,
    bf16 = bf16,
)

if RESPONSE_ONLY_LOSS:
    trainer = Trainer(
        model = model,
        args = training_args,
        train_dataset = train_dataset,
        data_collator = ResponseOnlyDataCollator(tokenizer, max_length=MAX_SEQ_LENGTH),
    )
else:
    trainer = build_sft_trainer(model, tokenizer, train_dataset, training_args)

trainer.train()

## 6. 保存 LoRA adapter / 合并模型

In [ ]:
SFT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 默认保存 LoRA adapter，后续可直接用这个目录推理或继续训练。
trainer.model.save_pretrained(str(SFT_OUTPUT_DIR))
tokenizer.save_pretrained(str(SFT_OUTPUT_DIR))
print("Saved LoRA adapter to:", SFT_OUTPUT_DIR)

# 如果需要部署合并模型，取消下面一行注释：
# trainer.model.save_pretrained_merged(str(SFT_OUTPUT_DIR) + "_merged_16bit", tokenizer, save_method="merged_16bit")

## 7. 单条推理

In [ ]:
FastLanguageModel.for_inference(trainer.model)


def generate_one(prompt: str) -> str:
    messages = [{"role": "user", "content": prompt}]
    text = _apply_chat_template(tokenizer, messages, add_generation_prompt=True)
    inputs = tokenizer([text], return_tensors="pt").to("cuda")
    outputs = trainer.model.generate(
        **inputs,
        max_new_tokens = MAX_NEW_TOKENS,
        temperature = TEMPERATURE,
        top_p = TOP_P,
        use_cache = True,
        pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id,
    )
    new_tokens = outputs[:, inputs.input_ids.shape[-1]:]
    return tokenizer.batch_decode(new_tokens, skip_special_tokens=True)[0].strip()

print(generate_one("请用一句话解释什么是大语言模型。"))

## 8. 批量推理（JSON / JSONL）

输入文件可以是：

- JSON array：`[{"prompt": "..."}, ...]`
- JSONL：每行一个 JSON object
- 每条数据也可以用 `messages` 字段代替 `prompt`

In [ ]:
def read_records(path: str | Path) -> list[dict[str, Any]]:
    raw = Path(path).read_text(encoding="utf-8").strip()
    if not raw:
        return []
    if Path(path).suffix.lower() == ".json" or raw.startswith("["):
        rows = json.loads(raw)
    else:
        rows = [json.loads(line) for line in raw.splitlines() if line.strip()]
    if not isinstance(rows, list) or not all(isinstance(row, dict) for row in rows):
        raise ValueError("Input must be a JSON array or JSONL of objects.")
    return rows


def write_records(path: str | Path, rows: list[dict[str, Any]]) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if path.suffix.lower() == ".jsonl":
        path.write_text("".join(json.dumps(row, ensure_ascii=False) + "\n" for row in rows), encoding="utf-8")
    else:
        path.write_text(json.dumps(rows, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def row_to_messages(row: dict[str, Any]) -> list[dict[str, str]]:
    if MESSAGES_FIELD in row:
        return row[MESSAGES_FIELD]
    return [{"role": "user", "content": row[PROMPT_FIELD]}]


def batch_generate(rows: list[dict[str, Any]]) -> list[dict[str, Any]]:
    results = [dict(row) for row in rows]
    pending: list[tuple[int, str]] = []
    for row_idx, row in enumerate(rows):
        text = _apply_chat_template(tokenizer, row_to_messages(row), add_generation_prompt=True)
        for _ in range(NUM_REPEATS):
            pending.append((row_idx, text))

    outputs_by_row = [[] for _ in rows]
    for start in range(0, len(pending), INFER_BATCH_SIZE):
        batch = pending[start:start + INFER_BATCH_SIZE]
        row_ids = [row_id for row_id, _ in batch]
        texts = [text for _, text in batch]
        inputs = tokenizer(texts, return_tensors="pt", padding=True).to("cuda")
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens = MAX_NEW_TOKENS,
            temperature = TEMPERATURE,
            top_p = TOP_P,
            use_cache = True,
            pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id,
        )
        new_tokens = outputs[:, inputs.input_ids.shape[-1]:]
        decoded = tokenizer.batch_decode(new_tokens, skip_special_tokens=True)
        for row_id, text in zip(row_ids, decoded):
            outputs_by_row[row_id].append(text.strip())
        print(f"Processed {min(start + len(batch), len(pending))}/{len(pending)} generations")

    for row, outs in zip(results, outputs_by_row):
        row["output"] = outs if NUM_REPEATS > 1 else outs[0]
    return results

prompt_rows = read_records(PROMPT_FILE)
batch_results = batch_generate(prompt_rows)
write_records(BATCH_OUTPUT_FILE, batch_results)
print("Wrote:", BATCH_OUTPUT_FILE)

## 9. 可选：GRPO / RL

默认不运行。你要做 RL 时再取消最后的 `grp_trainer.train()` 注释，并把 reward 函数替换成你的任务 reward。

In [ ]:
# from unsloth import PatchFastRL
# PatchFastRL("grpo", FastLanguageModel)
#
# from trl import GRPOConfig, GRPOTrainer
#
# rl_dataset = load_grpo_dataset(
#     TRAIN_FILE,
#     prompt_field = PROMPT_FIELD,
#     answer_field = RESPONSE_FIELD,
# )
#
# def reward_func(completions, answer, **kwargs):
#     rewards = []
#     for completion, expected in zip(completions, answer):
#         generated = completion if isinstance(completion, str) else str(completion)
#         rewards.append(1.0 if expected.strip().lower() in generated.strip().lower() else 0.0)
#     return rewards
#
# grpo_args = GRPOConfig(
#     output_dir = str(GRPO_OUTPUT_DIR),
#     learning_rate = 5e-6,
#     per_device_train_batch_size = 1,
#     gradient_accumulation_steps = 1,
#     num_generations = 2,
#     max_prompt_length = 768,
#     max_completion_length = 256,
#     max_steps = 100,
#     beta = 0.0,
#     fp16 = fp16,
#     bf16 = bf16,
#     logging_steps = 1,
#     report_to = "none",
# )
#
# grp_trainer = GRPOTrainer(
#     model = trainer.model,
#     processing_class = tokenizer,
#     reward_funcs = [reward_func],
#     args = grpo_args,
#     train_dataset = rl_dataset,
# )
#
# grp_trainer.train()
# grp_trainer.save_model(str(GRPO_OUTPUT_DIR))
# tokenizer.save_pretrained(str(GRPO_OUTPUT_DIR))

## 10. 对应脚本命令

Notebook 跑通后，建议用脚本跑正式实验：

```bash
# 指定物理 GPU 1 训练
CUDA_VISIBLE_DEVICES=1 python scripts/train_lora.py \
  --model_name_or_path model/Qwen3-1.7B \
  --train_file data/toy_sft.jsonl \
  --output_dir outputs/qwen3_1p7b_unsloth_lora

# 指定物理 GPU 1 批量推理
CUDA_VISIBLE_DEVICES=1 python scripts/batch_infer_lora.py \
  --model_name_or_path outputs/qwen3_1p7b_unsloth_lora \
  --input_file data/prompts.json \
  --output_file outputs/qwen3_1p7b_batch_outputs.json \
  --overwrite
```